# Cloud-edge experiment frontend
The sole editable configuration, including operational controls, is `config/experiment.json`. The current configuration enables preparation and submission of the approved full fresh rerun. Run All submits that graph once; subsequent submissions reuse its recorded identity. Scientific/source changes invalidate previous approvals; live guard changes do not. Resume requires explicit opt-in and submission permission.

## 1. Load and optionally prepare
Preparation creates the frozen plan only when `workflow.experiment.prepare_enabled` is explicitly enabled. Otherwise open an existing local experiment, or leave it unset.

In [1]:
from pathlib import Path
from cloud_edge_rl.config import load_config
from cloud_edge_rl.experiment import Experiment

PROJECT_ROOT = Path.cwd()
CONFIG = load_config(root=PROJECT_ROOT)
controls = CONFIG['workflow']['experiment']
experiment_directory = PROJECT_ROOT / controls['directory']
experiment = None
RESULTS = {}
if controls['prepare_enabled']:
    experiment = Experiment.prepare(CONFIG, PROJECT_ROOT, directory=experiment_directory)
elif experiment_directory.is_dir():
    experiment = Experiment.open(experiment_directory, root=PROJECT_ROOT)
print({'config': 'config/experiment.json', 'directory': str(experiment_directory), 'experiment_available': experiment is not None})

{'config': 'config/experiment.json', 'directory': 'c:\\DevSpace\\PyDevspace\\RL\\RL_project\\output\\fresh-seven-method-rerun-20261010-v1\\experiment', 'experiment_available': True}


## 2. Submit or explicitly resume
Submission permission is required for both operations. `resume_enabled` selects resume instead of submit; historical approvals do not authorize new execution. Scoped admission still applies.

In [2]:
if controls['submission_enabled']:
    if experiment is None:
        raise RuntimeError('Prepare or open an experiment before explicit submission')
    if controls.get('resume_enabled', False):
        RESULTS['submission'] = experiment.resume()
    else:
        RESULTS['submission'] = experiment.submit()
else:
    print('Submission disabled in config/experiment.json')

## 3. Status
The current configuration enables cloud status refresh. While the graph is running, collection and verification report pending. After cloud completion, rerun cells 3–6 to download and verify; do not enable resume for an already completed graph.

In [6]:
if experiment is not None:
    RESULTS['status'] = experiment.status(refresh=controls['refresh_enabled'])
else:
    RESULTS['status'] = {'state': 'not_prepared', 'directory': str(experiment_directory)}
print(RESULTS['status'])

{'experiment_id': 'experiment-07a6fb8d045b4b778c80b5fc4dba97de', 'planned_nodes': 101, 'status': 'blocked', 'formal_acceptance': 'not_evaluated', 'actual_cloud_cost_usd': None, 'submission': {'experiment_id': 'experiment-07a6fb8d045b4b778c80b5fc4dba97de', 'status': 'submitted', 'attempt_id': '121196f0242940a69c558b34a98ba2e3', 'call_id': 'fc-01M4K43PZHEHS1GK9HRFCHFQY7', 'app_id': 'ap-JfGxJcXz2xzLIrehlcGgCf', 'volume_name': 'cloud-edge-rl-fresh-rerun-20261010-v1', 'environment': 'main', 'reserved_run_ids': ['collect-188aa9a111a2422cb5e2b31f49a2fa0a', 'collect-809a4aaeafa743769e9184c87facf745', 'collect-17d67d674bb74abeb86efd471cea09ee', 'benchmark_attack-8796d9d23dab4f3d8e38f8b00fe8d5e6', 'audit-77a7df84741a44679bc9290c0f4994be', 'audit-79283c32d53444bda8359e34cb5f5594', 'audit-38650d98b91b481aaca08135dd4e5379', 'fit_risk-fc7e3f5c799f49c2bc74487687245745', 'train_policy-a5fde895481547fe9d145165b138cb18', 'train_policy-4d24ce0c1f1141c684df607b8d84324f', 'train_policy-006a5fc159aa4f689620

## 4. Collect
With explicit JSON opt-in, collect only after remote status confirms call completion. If export failed, download the already committed complete shards into `partial-artifacts/<attempt_id>/` using read-only Volume access; do not launch an export or resume compute. Partial downloads are not whole-experiment completion. Restart the kernel after code changes, run step 1, then steps 3–5; skip step 2 for this completed experiment.

In [9]:
if controls['collection_enabled']:
    remote_status = RESULTS['status'].get('remote-status', {})
    if experiment is not None and remote_status.get('call_completed') is True:
        available_only = (remote_status.get('status') == 'blocked'
                          and remote_status.get('error', {}).get('phase') == 'export')
        if available_only:
            print('Downloading committed shards only; no cloud compute. Whole-experiment transfer remains incomplete.')
            RESULTS['collection'] = experiment.collect(available_only=True)
        else:
            RESULTS['collection'] = experiment.collect()
        RESULTS['status'] = experiment.status(refresh=False)
        print(RESULTS['collection'])
    else:
        print('Collection pending: remote call completion not confirmed')
else:
    print('Collection disabled in config/experiment.json')

## 5. Verify
With explicit JSON opt-in, verify only a transport-complete artifact transfer bound to the current submission attempt and experiment. Otherwise report pending.

In [10]:
if controls['verification_enabled']:
    local_status = RESULTS['status']
    transfer = local_status.get('transfer', {})
    submission = local_status.get('submission', {})
    if (experiment is not None and transfer.get('transport_complete') is True
            and submission.get('attempt_id') is not None
            and transfer.get('attempt_id') == submission['attempt_id']
            and transfer.get('experiment_id') == local_status.get('experiment_id')):
        RESULTS['verification'] = experiment.verify()
    else:
        partial = local_status.get('partial-transfer', {})
        if (partial.get('available_shards_complete') is True
                and partial.get('attempt_id') == submission.get('attempt_id')):
            print('Partial download available at:', partial['destination'])
        print('Verification pending: current-attempt artifact transfer not complete')
else:
    print('Verification disabled in config/experiment.json')

Verification pending: current-attempt artifact transfer not complete


## 6. Results
Display returned status and artifacts only; this cell performs no inference, submission, or implicit resume.

In [ ]:
print(RESULTS)